# VoiceForge Auto-Train

Train an RVC v2 voice model on Google Colab with one click.

**If opened from VoiceForge app:** Just click **Runtime → Run all**. Everything auto-detects from Drive.

**If opened manually:**
1. In VoiceForge, upload audio and click **Start Training** — data goes to Drive automatically
2. Open this notebook and click **Runtime → Run all**
3. Model name auto-detects from your Drive folder

Runtime → Change runtime type → **T4 GPU**

## 1. Configure

In [ ]:
#@title Training Settings

GDRIVE_FILE_ID = "" #@param {type:"string"}
MODEL_NAME = "my_voice" #@param {type:"string"}

RVC_DIR = "/content/RVC-WebUI"
EXP_DIR = f"{RVC_DIR}/logs/{MODEL_NAME}"
ASSETS_DIR = "/content/assets"
DATASET_DIR = "/content/dataset"

print(f"Model: {MODEL_NAME}")
print(f"Exp dir: {EXP_DIR}")
if GDRIVE_FILE_ID:
    print(f"Drive File ID: {GDRIVE_FILE_ID}")

In [ ]:
#@title Mount Google Drive (for saving trained model)
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
#@title Auto-detect model from Drive (if you used VoiceForge app)
import glob
import os

# If MODEL_NAME still default, try auto-detecting from Drive/VoiceForge/
if MODEL_NAME == "my_voice" and os.path.exists("/content/drive/MyDrive/VoiceForge"):
    folders = sorted(glob.glob("/content/drive/MyDrive/VoiceForge/*/"),
                     key=os.path.getmtime, reverse=True)
    if folders:
        detected = os.path.basename(folders[0].rstrip("/"))
        MODEL_NAME = detected
        EXP_DIR = f"{RVC_DIR}/logs/{MODEL_NAME}"
        print(f"Auto-detected model: {MODEL_NAME}")

print(f"Using model: {MODEL_NAME}")
print(f"Exp dir: {EXP_DIR}")

## 2. Setup Environment

In [ ]:
#@title Install system dependencies
!apt-get -qq update
!apt-get -qq install -y libsndfile1-dev ffmpeg unzip wget curl
print("System deps installed")

In [ ]:
#@title Download training dataset from Google Drive

import shutil
import glob
import os, zipfile

dataset_zip = "/content/dataset.zip"

if GDRIVE_FILE_ID:
    !gdown --id {GDRIVE_FILE_ID} -O {dataset_zip} --fuzzy
    print(f"Downloaded: {dataset_zip}")
elif MODEL_NAME:
    drive_pattern = f"/content/drive/MyDrive/VoiceForge/{MODEL_NAME}/*.zip"
    zips = sorted(glob.glob(drive_pattern), key=os.path.getmtime, reverse=True)
    if zips:
        shutil.copy(zips[0], dataset_zip)
        print(f"Auto-detected zip from Drive: {zips[0]}")
    else:
        print(f"No zip found at Drive/VoiceForge/{MODEL_NAME}/")
        print("Falling back to manual upload...")
        from google.colab import files
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        !unzip -o {zip_name} -d {DATASET_DIR}
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -o {zip_name} -d {DATASET_DIR}

if os.path.exists(dataset_zip):
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(dataset_zip, "r") as zf:
        zf.extractall(DATASET_DIR)
    print(f"Extracted to: {DATASET_DIR}")
    !ls -la {DATASET_DIR}

In [ ]:
#@title Clone RVC WebUI
if not os.path.exists(RVC_DIR):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print("RVC already cloned")
%cd {RVC_DIR}
print(f"Working dir: {RVC_DIR}")

In [ ]:
#@title Install Python dependencies (skip problematic packages)
!pip install -q --no-deps pyworld
!pip install -q numpy==1.23.5 numba==0.56.4 llvmlite==0.39.0
!pip install -q librosa==0.9.1 soundfile ffmpeg-python pydub
!pip install -q faiss-cpu>=1.8.0 gradio==3.34.0 scipy
!pip install -q tensorboardX matplotlib
!pip install -q gdown
!pip install -q praat-parselmouth

# fairseq often fails - skip, not needed for v2 training
!pip install -q Cython
print("Python deps installed")

In [ ]:
#@title Download pretrained models
os.makedirs(f"{ASSETS_DIR}/hubert", exist_ok=True)
os.makedirs(f"{ASSETS_DIR}/rmvpe", exist_ok=True)
os.makedirs(f"{ASSETS_DIR}/pretrained", exist_ok=True)

if not os.path.exists(f"{ASSETS_DIR}/hubert/hubert_base.pt"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt -O {ASSETS_DIR}/hubert/hubert_base.pt

if not os.path.exists(f"{ASSETS_DIR}/rmvpe/rmvpe.pt"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt -O {ASSETS_DIR}/rmvpe/rmvpe.pt

if not os.path.exists(f"{ASSETS_DIR}/pretrained/f0G40k.pth"):
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth -O {ASSETS_DIR}/pretrained/f0G40k.pth
    !wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth -O {ASSETS_DIR}/pretrained/f0D40k.pth

print("Pretrained models ready")

## 3. Preprocess Dataset

In [ ]:
#@title Step 3a: Resample audio to 16kHz
%cd {RVC_DIR}
!python infer/modules/train/preprocess.py {DATASET_DIR} 16000 4 {EXP_DIR} False 3.0
print("Resampling complete")

In [ ]:
#@title Step 3a.5: Prepare training config + filelist
import glob, json, shutil, os

os.makedirs(EXP_DIR, exist_ok=True)

# Copy config template (32k for 16kHz input training)
config_src = f"{RVC_DIR}/configs/v2/32k.json"
config_dst = f"{EXP_DIR}/config.json"
if os.path.exists(config_src):
    shutil.copy(config_src, config_dst)
    print(f"Config copied: {config_dst}")
else:
    print(f"WARNING: Config template not found at {config_src}")

# Generate filelist.txt for training data loader
wav_dir = f"{EXP_DIR}/1_16k_wavs"
if os.path.exists(wav_dir):
    wav_files = sorted(glob.glob(f"{wav_dir}/*.wav"))
    with open(f"{EXP_DIR}/filelist.txt", "w") as f:
        for w in wav_files:
            f.write(w + "\n")
    print(f"filelist.txt generated: {len(wav_files)} files")
else:
    print("WARNING: 1_16k_wavs not found yet — run Step 3a first")

In [ ]:
#@title Step 3b: Extract pitch (f0)
!python infer/modules/train/extract/extract_f0_print.py {EXP_DIR} 4 rmvpe
print("F0 extraction complete")

In [ ]:
#@title Step 3c: Extract hubert features
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 {EXP_DIR} v2 True
print("Feature extraction complete")

## 4. Train Model (30-60 min on T4 GPU)

In [ ]:
#@title Start training
%cd {RVC_DIR}
!python infer/modules/train/train.py     -e {MODEL_NAME}     -sr 32k     -v v2     -f0 1     -bs 4     -g 0     -te 100     -se 20     -l 0     -c 0     -sw 0     -pg {ASSETS_DIR}/pretrained/f0G40k.pth     -pd {ASSETS_DIR}/pretrained/f0D40k.pth
print("Training complete!")

In [ ]:
#@title Step 4b: Generate index file
%cd {RVC_DIR}
!python tools/infer/train-index.py -d {EXP_DIR} -n {EXP_DIR}/{MODEL_NAME}.index
print("Index file generated")

## 5. Export Model to Google Drive

In [ ]:
#@title Save .pth and .index to your Drive
import shutil
import glob
import os

drive_dir = f"/content/drive/MyDrive/VoiceForge/{MODEL_NAME}"
os.makedirs(drive_dir, exist_ok=True)

pth_files = sorted(glob.glob(f"{EXP_DIR}/*.pth") + glob.glob(f"{EXP_DIR}/checkpoints/*.pth"),
                   key=os.path.getmtime, reverse=True)
if pth_files:
    shutil.copy(pth_files[0], f"{drive_dir}/{MODEL_NAME}.pth")
    filesize = os.path.getsize(f"{drive_dir}/{MODEL_NAME}.pth") / 1024 / 1024
    print(f"Copied .pth ({filesize:.1f} MB): {pth_files[0]}")
else:
    print("WARNING: No .pth file found!")

index_files = glob.glob(f"{EXP_DIR}/*.index") + glob.glob(f"{EXP_DIR}/index/*.index")
for f in index_files:
    shutil.copy(f, f"{drive_dir}/{MODEL_NAME}.index")
    print(f"Copied .index: {f}")

print(f"
Model saved to: {drive_dir}")

## Done!

1. Model saved to **Google Drive → VoiceForge/{MODEL_NAME}/**
2. Go back to **VoiceForge app → Train page**
3. Click **Import Model** — it downloads .pth and .index automatically
4. Model appears on **Convert** and **TTS** pages

**Manual alternative:** Download .pth + .index from Drive and drop into VoiceForge → models folder.